In [1]:
import gzip
import os
import urllib.request

import numpy as np
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer

# The course's automatic tests set COURSE_SMOKE=1. This notebook is already small, so the
# only change is fewer GloVe words (the 50,000 most common instead of 200,000).
SMOKE = os.environ.get("COURSE_SMOKE") == "1"
torch.manual_seed(0)

## 1. Static word vectors: GloVe

A static embedding is just a lookup table: each word maps to one row of numbers.
GloVe was trained on word co-occurrence counts from Wikipedia and newswire text. This set has 400,000 words and 50 dimensions.
The file is plain text, one word per line, most common words first. We keep the most common ones.

In [2]:
GLOVE_URL = "https://github.com/piskvorky/gensim-data/releases/download/glove-wiki-gigaword-50/glove-wiki-gigaword-50.gz"
GLOVE_FILE = "glove-wiki-gigaword-50.gz"
N_WORDS = 50_000 if SMOKE else 200_000

if not os.path.exists(GLOVE_FILE):
    urllib.request.urlretrieve(GLOVE_URL, GLOVE_FILE)

words, rows = [], []
with gzip.open(GLOVE_FILE, "rt", encoding="utf-8") as f:
    next(f)  # the first line is a header: "400000 50"
    for line, _ in zip(f, range(N_WORDS)):
        word, *numbers = line.rstrip().split(" ")
        words.append(word)
        rows.append(np.array(numbers, dtype=np.float32))

vectors = np.stack(rows)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)  # unit length, so a dot product is a cosine
index = {w: i for i, w in enumerate(words)}
print(vectors.shape)
print("king:", np.round(vectors[index["king"]][:8], 2), "...")

(200000, 50)
king: [ 0.09  0.13 -0.11 -0.    0.11 -0.03 -0.02  0.09] ...


In [3]:
def nearest(vector, exclude=(), k=5):
    """The k words whose vectors have the highest cosine similarity with `vector`."""
    vector = vector / np.linalg.norm(vector)
    scores = vectors @ vector
    best = []
    for i in np.argsort(-scores):
        if words[i] not in exclude:
            best.append((words[i], round(float(scores[i]), 3)))
        if len(best) == k:
            return best


def analogy(a, b, c):
    """a is to b as c is to ?  Computes b - a + c, e.g. king - man + woman."""
    v = vectors[index[b]] - vectors[index[a]] + vectors[index[c]]
    return nearest(v, exclude={a, b, c})


print("man : king :: woman : ?", analogy("man", "king", "woman"))

man : king :: woman : ? [('queen', 0.852), ('throne', 0.766), ('prince', 0.759), ('daughter', 0.747), ('elizabeth', 0.746)]


The answer leaves out the three input words, as Mikolov et al. (2013) did. Without that rule, the closest word to king − man + woman is usually "king" itself.
The arithmetic works because the offset man → king points roughly the same way as woman → queen.

More analogies. Some work and some do not: the vectors only learned what the text showed them.

In [4]:
for a, b, c in [("france", "paris", "italy"),     # capital cities
                ("walk", "walking", "swim"),      # verb forms
                ("good", "better", "bad"),        # comparatives
                ("canada", "ottawa", "japan"),
                ("man", "doctor", "woman")]:      # a bias check
    print(f"{a} : {b} :: {c} : ?", analogy(a, b, c)[:3])

france : paris :: italy : ? [('rome', 0.847), ('milan', 0.777), ('turin', 0.767)]
walk : walking :: swim : ? [('swimming', 0.807), ('swimmers', 0.75), ('swims', 0.745)]
good : better :: bad : ? [('worse', 0.901), ('too', 0.832), ('unfortunately', 0.822)]


canada : ottawa :: japan : ? [('osaka', 0.77), ('yokohama', 0.745), ('nagoya', 0.744)]


man : doctor :: woman : ? [('nurse', 0.84), ('child', 0.766), ('pregnant', 0.757)]


Paris, swimming and worse come out right. Ottawa to Japan gives Osaka, though Tokyo is the capital. The offsets only roughly line up, so the nearest word is sometimes a close cousin of the right one.
Doctor for a man gives nurse for a woman. The vectors learned a stereotype from the text they were trained on ([Bolukbasi et al., 2016](https://arxiv.org/abs/1607.06520)).

In [5]:
cosine = lambda u, v: float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))
print("king vs queen:", round(cosine(vectors[index["king"]], vectors[index["queen"]]), 3))
print("king vs banana:", round(cosine(vectors[index["king"]], vectors[index["banana"]]), 3))

# One vector per word, whatever the sentence: "bank" means the same thing everywhere
print("nearest to 'bank':", nearest(vectors[index["bank"]], exclude={"bank"}))

king vs queen: 0.784
king vs banana: 0.221
nearest to 'bank': [('banks', 0.87), ('securities', 0.8), ('banking', 0.797), ('investment', 0.785), ('exchange', 0.781)]


The neighbours of "bank" are all about money. A static vector is an average over every use of the word in the training text, so the river meaning is drowned out.

## 2. Contextual token vectors: BERT

BERT splits the text into tokens and gives each token a vector. Each of its 12 layers rebuilds every token's vector by attending to the other tokens in the sentence.
So the same word gets a different vector in each sentence.

In [6]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert = AutoModel.from_pretrained("google-bert/bert-base-uncased")
bert.eval()

sentence = "The fisherman sat on the river bank."
inputs = tokenizer(sentence, return_tensors="pt")
with torch.no_grad():
    hidden = bert(**inputs).last_hidden_state[0]  # (tokens, 768): one vector per token

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(len(tokens), "tokens:", tokens)
print("vectors:", tuple(hidden.shape))

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


10 tokens: ['[CLS]', 'the', 'fisherman', 'sat', 'on', 'the', 'river', 'bank', '.', '[SEP]']
vectors: (10, 768)


In [7]:
def token_vector(sentence, word):
    """BERT's last-layer vector for `word` (a single token) in `sentence`."""
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    with torch.no_grad():
        hidden = bert(**inputs).last_hidden_state[0]
    return hidden[tokens.index(word)]


sentences = {
    "river 1": "The fisherman sat on the river bank.",
    "river 2": "Reeds grow along the muddy bank of the stream.",
    "money 1": "I deposited my pay at the bank.",
    "money 2": "The bank approved the loan for our new house.",
}
bank = {name: token_vector(s, "bank") for name, s in sentences.items()}

names = list(bank)
print(f"{'cosine':>8}" + "".join(f"{n:>9}" for n in names))
for a in names:
    print(f"{a:>8}" + "".join(f"{F.cosine_similarity(bank[a], bank[b], dim=0).item():9.2f}" for b in names))

  cosine  river 1  river 2  money 1  money 2
 river 1     1.00     0.80     0.51     0.42
 river 2     0.80     1.00     0.41     0.37
 money 1     0.51     0.41     1.00     0.78
 money 2     0.42     0.37     0.78     1.00


The two river sentences give "bank" vectors that are closer to each other than to the money sentences, and the other way round.
BERT has read the sentence around the word. That is what "contextual" means.

**And king − man + woman?** BERT has no single vector for "king": each sentence gives a new one.
The original version of this demo put each word in a neutral sentence ("This is a king.") and took the `[CLS]` vector, a summary of the whole sentence.

In [8]:
def cls_vector(word, template="This is a {}."):
    inputs = tokenizer(template.format(word), return_tensors="pt")
    with torch.no_grad():
        return F.normalize(bert(**inputs).last_hidden_state[0, 0], dim=0)  # [CLS] is the first token


demo_words = ["man", "woman", "king", "queen", "banana", "carburetor"]
cls = {w: cls_vector(w) for w in demo_words}
target = cls["king"] - cls["man"] + cls["woman"]
for w in demo_words:
    print(f"{w:>11}: {F.cosine_similarity(target, cls[w], dim=0).item():.3f}")

        man: 0.922
      woman: 0.971
       king: 0.972
      queen: 0.985
     banana: 0.944
 carburetor: 0.837


"Queen" comes out on top, but every word scores above 0.8, and "banana" beats "man".
The `[CLS]` vectors of these near-identical sentences all point the same way, so the analogy barely separates the right answer from nonsense.
Raw BERT vectors are a poor fit for similarity search. That is why Sentence-BERT (Reimers and Gurevych, 2019) was trained, which brings us to part 3.

## 3. Sentence embeddings and cosine similarity

A sentence-embedding model pools the token vectors into one vector per sentence (here, their mean).
It was trained on pairs of sentences that mean the same thing, so similar meanings land close together.

In [9]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
docs = [
    "The fisherman sat on the river bank.",
    "Reeds grow along the muddy bank of the stream.",
    "I deposited my pay at the bank.",
    "The bank approved the loan for our new house.",
    "How do I open a savings account?",
    "Where can I go fishing near the water?",
]
embeddings = encoder.encode(docs, normalize_embeddings=True)  # unit length
print(embeddings.shape)  # one 384-number vector per sentence, however long the sentence

similarity = embeddings @ embeddings.T  # cosine similarity, since the vectors have unit length
for i, d in enumerate(docs):
    print(f"{d[:46]:<47}" + " ".join(f"{s:5.2f}" for s in similarity[i]))

(6, 384)
The fisherman sat on the river bank.            1.00  0.27  0.24  0.18  0.14  0.37
Reeds grow along the muddy bank of the stream.  0.27  1.00  0.09  0.05  0.04  0.28
I deposited my pay at the bank.                 0.24  0.09  1.00  0.31  0.30  0.07
The bank approved the loan for our new house.   0.18  0.05  0.31  1.00  0.21 -0.06
How do I open a savings account?                0.14  0.04  0.30  0.21  1.00  0.08
Where can I go fishing near the water?          0.37  0.28  0.07 -0.06  0.08  1.00


A tiny semantic search: embed the question, then rank the sentences by cosine similarity.
The best match shares meaning with the question, even with few words in common.

In [10]:
for question in ["Who lends money for a mortgage?", "Where do plants grow by a creek?"]:
    q = encoder.encode([question], normalize_embeddings=True)[0]
    scores = embeddings @ q
    best = np.argsort(-scores)[:2]
    print(question)
    for i in best:
        print(f"  {scores[i]:.2f}  {docs[i]}")

Who lends money for a mortgage?
  0.54  The bank approved the loan for our new house.
  0.24  How do I open a savings account?
Where do plants grow by a creek?
  0.54  Reeds grow along the muddy bank of the stream.
  0.39  Where can I go fishing near the water?


## Try it

- Find an analogy that GloVe gets wrong, and explain why from how the vectors were trained.
- Put "bank" in a sentence that could mean either thing ("We walked to the bank."). Which group is its vector closer to?
- Swap `all-MiniLM-L6-v2` for another model from the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) and compare the similarity table.